# 02 — Pré-processamento

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [10]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

RANDOM_STATE = 42

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path("..")
else:
    PROJECT_ROOT = Path(".")

RAW_APP = PROJECT_ROOT / "data" / "raw" / "application_record.csv"
RAW_CREDIT = PROJECT_ROOT / "data" / "raw" / "credit_record.csv"
PROCESSED = PROJECT_ROOT / "data" / "processed"
TARGET = "target"

pd.set_option("display.max_columns", None)

In [11]:
# Repetindo a lógica para criar o dataset base
df_app = pd.read_csv(RAW_APP)
df_credit = pd.read_csv(RAW_CREDIT)

# 1. Definição de Mau Pagador: Atraso superior a 60 dias (status '2', '3', '4', '5')
bad_status = ['2', '3', '4', '5']
df_credit['is_bad'] = df_credit['STATUS'].isin(bad_status).astype(int)
# 2. Agregação no histórico: Target (calote histórico) + Tempo de conta observado (Account Age)
credit_summary = df_credit.groupby('ID').agg(
    target=('is_bad', 'max'),
    ACCOUNT_AGE_MONTHS=('MONTHS_BALANCE', lambda s: abs(s.min()))
).reset_index()

# 3. Inner join para manter apenas clientes com aplicação e histórico de pagamento
df = df_app.merge(credit_summary, on='ID', how='inner')
print(f"Base após merge: {df.shape[0]} proponentes com histórico de crédito.")

Base após merge: 36457 proponentes com histórico de crédito.


## 1. Dados faltantes

In [12]:
nulos = df.isna().sum()
pd.DataFrame({"nulos": nulos, "pct": (nulos / len(df) * 100).round(2)}).query("nulos > 0")

,nulos,pct
OCCUPATION_TYPE,11323,31.06


In [13]:
# A principal variável com nulos é OCCUPATION_TYPE
# Preencher com "Desconhecido" é mais seguro do que descartar as linhas
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Unknown')

**Decisão:** _preencher._

## 2. Definição da variável alvo

_Se houve binarização, qual limiar e por quê? Justifique com base na distribuição, não por convenção._

In [14]:
# ex.: df['alvo'] = (df[TARGET] >= LIMIAR).astype(int)

## 3. Normalização / padronização

**Escolha e justificativa:** _StandardScaler, MinMaxScaler ou nenhum?_

Aplique preferencialmente dentro de um `Pipeline` no notebook 03 — ajustar o scaler antes do split vaza informação do teste para o treino.

In [15]:
# escala

## 4. Feature engineering

**Justificativa:** _preencher._

In [16]:
# novas variáveis derivadas

In [17]:
# --- 1. Idade e Tempo de Emprego ---
df['AGE_YEARS'] = (df['DAYS_BIRTH'] / -365.25).astype(int)
df['FLAG_UNEMPLOYED'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['YEARS_EMPLOYED'] = np.where(df['DAYS_EMPLOYED'] == 365243, 0.0, df['DAYS_EMPLOYED'] / -365.25)

# --- 2. Ratios Econômicos e de Estabilidade (Credit Scoring Ratios) ---
# Renda per capita (muito mais explicativa sobre capacidade de pagamento que renda total)
df['INCOME_PER_CAPITA'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

# Estabilidade profissional: fração da vida adulta em que esteve empregado
adult_years = np.maximum(df['AGE_YEARS'] - 18, 1)
df['EMPLOYED_TO_AGE_RATIO'] = (df['YEARS_EMPLOYED'] / adult_years).clip(0, 1)

# Idade estimada em que começou a trabalhar
df['START_WORK_AGE'] = (df['AGE_YEARS'] - df['YEARS_EMPLOYED']).clip(lower=14)

# Renda por ano de experiência no emprego atual
df['INCOME_PER_YEAR_EMPLOYED'] = df['AMT_INCOME_TOTAL'] / (df['YEARS_EMPLOYED'] + 1)

# Logaritmo da renda para desinflar a cauda longa
df['LOG_INCOME'] = np.log1p(df['AMT_INCOME_TOTAL'])

# --- 3. Score de Patrimônio Próprio ---
df['FLAG_OWN_CAR'] = (df['FLAG_OWN_CAR'] == 'Y').astype(int)
df['FLAG_OWN_REALTY'] = (df['FLAG_OWN_REALTY'] == 'Y').astype(int)
df['PATRIMONY_SCORE'] = df['FLAG_OWN_CAR'] + df['FLAG_OWN_REALTY']

# Mora sozinho
df['IS_ALONE'] = (df['CNT_FAM_MEMBERS'] == 1).astype(int)
df['IS_MALE'] = (df['CODE_GENDER'] == 'M').astype(int)

# --- 4. Descarte de Colunas Redundantes, Variância Zero ou Ruído ---
cols_to_drop = [
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'CODE_GENDER',
    'FLAG_MOBIL',       # Variância zero (100% dos clientes têm celular)
    'FLAG_WORK_PHONE',  # Baixíssimo poder preditivo / ruído
    'FLAG_PHONE', 'FLAG_EMAIL',
    'CNT_CHILDREN'      # Multicolinearidade quase perfeita com CNT_FAM_MEMBERS
]
df = df.drop(columns=cols_to_drop)

# --- 5. Codificação Categórica via One-Hot Encoding Limpo ---
cat_cols = ['NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print(f"Colunas finais após Feature Engineering: {df.shape[1]}")

Colunas finais após Feature Engineering: 53


## 5. Salvar dataset tratado

In [18]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED / "dataset_tratado.csv", index=False)
print(f"Dataset com shape: {df.shape}")
print(f"Dataset tratado salvo com sucesso em: {PROCESSED / 'dataset_tratado.csv'}")

Dataset com shape: (36457, 53)
Dataset tratado salvo com sucesso em: data/processed/dataset_tratado.csv
